# Self-RAG agent

[Large language models (LLMs)](https://www.ibm.com/think/topics/large-language-models) have remarkable text generation and reasoning abilities but often produce factual inaccuracies or [hallucinations](https://www.ibm.com/think/topics/ai-hallucinations) due to their reliance on internal knowledge. [Retrieval augmented generation](https://www.ibm.com/think/topics/retrieval-augmented-generation) (RAG) based solutions aim to resolve this by injecting external documents into the model’s context. However, traditional [RAG approaches](https://www.ibm.com/think/topics/rag-techniques) retrieve a fixed number of passages regardless of their necessity or quality, leading to redundancy, inefficiency, and inconsistent factual grounding. 

The self-RAG framework provides a practical solution to this problem. It retrieves information on-demand by using special control tokens that dynamically decide when and how to perform retrieval during generation. Unlike agentic or multi-agent approaches that coordinate multiple models or components, self-RAG is a model-centric framework where a single model manages retrieval, generation, and critique internally. Its self-critique process is a structured step where the model evaluates both its own output and the quality of the retrieved information, allowing it to adapt its retrieval behavior through self-reflection tokens. It combines retrieval, generation and self-critique of its own generations with a single model trained end-to-end that allows more efficient, factual and controllable text generation. This method was originally introduced in the paper on *Self-RAG: Learning to Retrieve, Generate, and Critique Through Self-Reflection* (2024), which explores how [fine-tuning](https://www.ibm.com/think/topics/fine-tuning) LLMs for self-evaluation can improve factual consistency in [natural language processing](https://www.ibm.com/think/topics/natural-language-processing) (NLP) tasks.


## How self-RAG works

The workflow of self-RAG is orchestrated by special reflection tokens that the model generates alongside its text output, making the entire inference process dynamic and controllable. When additional information is needed, a single LLM takes on both the retriever and critic roles. A retriever component fetches relevant external passages, and the same LLM then uses reflection tokens to evaluate and refine its own generation during inference. This architecture represents a broader trend in [artificial intelligence](https://www.ibm.com/think/artificial-intelligence) (AI) toward models capable of introspection and dynamic reasoning, bridging advances in prompt engineering and long-form generations.

### 1. On-demand retrieval

The LLM first generates a retrieval token to determine whether external factual information is necessary for the query. The model skips the remaining retrieval-based steps and continues with standard generation if it concludes that retrieval is not necessary. If the retrieval token is decoded as “yes”, a retriever is called to fetch a set of relevant passages from an external knowledge base. This step makes sure that retrieval occurs when its expected utility is high.

### 2. Passage retrieval and generation

If [retrieval](https://www.ibm.com/think/topics/information-retrieval) is required, the retriever fetches relevant passages from an external knowledge base. The LLM simultaneously processes the input and retrieved passages and generates text continuation for each passage.

### 3. Generate and reflect on retrieved passages

For each segment generated, the model concurrently generates special critique tokens that are embedded directly within the output sequence. These tokens are not separate evaluations, rather they appear as part of the generated sequence and help the model check its own work as it goes:

**ISREL (Relevance):** Assesses the usefulness of the retrieved passage.

**ISSUP (Support/Factuality):** Evaluates if the generated text segment has whole, partial, or no factual support from the source material.

**ISUSE (Utility):** Evaluates the created segment's overall quality, usefulness, and structure.


### 4. Inference

During inference, reflection tokens ae used to decide when to retrieve information or not. It enables the model to adjust to different tasks, such as retrieving less for creative activities and more for factual ones. When generating text, reflection tokens help the model in adhering to particular guidelines. They either provide clear boundaries or guide word choice, which makes the model's responses more flexible and appropriate for various contexts.

### 5. Training the self-RAG

During training, reflection tokens are inserted into the [training data](https://www.ibm.com/think/topics/training-data) based on evaluations made by the critic model. This approach keeps self-rag training efficient by allowing the model to learn how to judge its own outputs and decide when it actually needs to look up information. Hence, the model becomes better at producing accurate, controlled, and high-quality responses.

In the experiment conducted in the research mentioned previously, self-RAG outperforms many standard retrieval-augmented and instruction-tuned baselines across various tasks, including open-domain [question answering](https://www.ibm.com/think/topics/question-answering), reasoning, and fact verification. It improves factuality and citation accuracy by using self-reflection tokens and on-demand retrieval.

This notebook focuses on building a robust self-reflective [RAG agent](https://www.ibm.com/think/topics/agentic-rag) by using OpenAI's [GPT-4o family of models](https://platform.openai.com/docs/models) and [LangGraph](https://www.ibm.com/think/topics/langgraph). Similar frameworks and tools, such as LlamaIndex or LangChain, also enable complex RAG flows. This tutorial focuses on using OpenAI's multi-modal `gpt-4o-mini` model, which understands both text and images natively. Combined with OpenAI's `text-embedding-3-small` model for retrieval, this stack offers a cost-effective, high-quality foundation for building reliable RAG systems that can handle complex data with strong factual grounding.


## Use case: Building a self-RAG query agent over multi-modal documents

This tutorial demonstrates how to build a self-RAG agent designed to answer complex, multi-faceted queries over internal knowledge bases that include both text and visual data. This agent will analyze PDF documents including technical guidelines and survey data. It will guide you to implement the self-RAG algorithm, which:

**Creates a multi-modal knowledge base:** Uses OpenAI's `gpt-4o-mini` (which natively handles both text and images) to extract text and images from PDFs and generate descriptive captions, and uses OpenAI's `text-embedding-3-small` to create embeddings for both text and image captions to enable semantic retrieval.

**Generates and reflects:** It creates an answer segment, adds reflection tokens (such ISREL, ISSUP, and ISUSE) and evaluates its own output quality and factual accuracy.

**Executes self-correction:** The LangGraph workflow extends the standard self-RAG approach by using a critique score derived from reflection tokens to guide its next steps. When the score is low, the agent requests stronger context and retrieves more relevant information before generating the next segment, helping produce a higher-quality final output.

**Provides segmented answers:** Provides thorough and traceable responses by generating complex answers in a sequence of factually validated chunks.


## Prerequisites

You need an [OpenAI account](https://platform.openai.com/signup) to obtain an OpenAI API key for accessing the GPT models. Make sure your account has access to the chat completion models you plan to use (for example, `gpt-4o-mini` for text and vision, plus `text-embedding-3-small` for embeddings).

## Steps

### Step 1. Set up your environment

While you can choose from several tools, this tutorial walks you through running the notebook in a local Jupyter environment.

1.	Create a Python virtual environment (e.g., with `conda` or `venv`).

2.	Launch a [Jupyter Notebook](https://jupyter.org/install) server and open this notebook.

3.	Store your `OPENAI_API_KEY` in a `.env` file in the same directory as this notebook (or export it as an environment variable). The notebook will load it automatically via `python-dotenv`.

**Note:** No GPU is required. All heavy lifting (LLM reasoning, vision captioning, and embeddings) is performed remotely by OpenAI, so even modest hardware works well.


### Step 2. Set up your OpenAI API key

1.	Sign in to the [OpenAI platform](https://platform.openai.com/) with your account.

2.	Navigate to the [API keys page](https://platform.openai.com/api-keys) and click "Create new secret key".

3.	Copy the key and store it in a `.env` file as `OPENAI_API_KEY=sk-...` in the same folder as this notebook.


### Step 3. Installation of the packages

To build and orchestrate this multi-modal self-reflective RAG agent, we require a comprehensive set of libraries. Install `langgraph` to define the core state machine that orchestrates the self-correction loop based on critique ratings. For integrating OpenAI chat models and [embeddings](https://www.ibm.com/think/topics/embedding), install `langchain-openai`. For quick retrieval, install `faiss-cpu` that offers indexing for the vector store. To extract and process the text and images from our PDF documents, `pillow` and `pymupdf` are essential. Lastly, install `python-dotenv` to securely load the `OPENAI_API_KEY` from a local `.env` file.

In [ ]:
# Install packages

!pip install -U langchain langchain-openai langchain-community \
  langchain-text-splitters langgraph faiss-cpu Pillow pymupdf \
  python-dotenv pydantic

print("Required packages installed.")

**Note:** No GPU is required, but execution can be slower on CPU-based systems. 

### Step 4. Import required libraries

Next, import all the necessary modules to set up the fundamental tools for managing the multi-modal components, processing documents, coordinating the RAG [workflow](https://www.ibm.com/think/topics/agentic-workflows), and connecting to OpenAI.

In [ ]:
# Core Libraries Import

import os
import re
import io
import gc
import base64
import getpass
from pathlib import Path
from typing import List, Dict, Any, TypedDict

# LangGraph / LangChain Core
from langgraph.graph import StateGraph, END, START
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Vector store + text utilities
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# General utils
from tqdm import tqdm
from PIL import Image
import fitz  # PyMuPDF for PDFs
import numpy as np

# Load environment variables (e.g. OPENAI_API_KEY) from a local .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("Core libraries imported successfully.")

**Multi-modal context:** This tutorial uses OpenAI's `gpt-4o-mini` (which natively understands images) along with `pymupdf` to process both text and visual data into a unified context. This surpasses simple text-based RAG by enabling the agent to retrieve richer information and provide highly accurate answers derived from complex documents.

**Self-correction loop:** The system utilizes LangGraph (StateGraph) to build a self-reflective RAG agent. This allows the LLM to critique its own output for relevance and accuracy, and then automatically initiate a correction cycle by querying the vector store or refining the prompt, minimizing hallucinations.

**Production-ready integration:** The tutorial demonstrates a high-performance stack by integrating OpenAI's hosted LLMs (accessed via the OpenAI [API](https://www.ibm.com/think/topics/api)) with efficient vector storage (FAISS) and streamlined RAG logic, proving its viability for real-world deployment.


### Step 5. Load OpenAI credentials

This step prepares your environment to securely connect to the OpenAI platform, allowing you to utilize the hosted GPT-4o models and embeddings.

In [ ]:
# Load OpenAI Credentials

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API Key: ")

# Re-export so libraries that read from the environment pick it up
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI credentials loaded.")

### Step 6. Initialize models

This step configures the two OpenAI models required for our multi-modal self-RAG agent: a chat model for reasoning, generation, self-critique, and image captioning, plus an embedding model for retrieval.

In [ ]:
# Initialize Models

# Chat LLM: gpt-4o-mini (used as both the generator/critic AND the vision captioner).
# Lower temperature keeps the self-critique loop deterministic and factual.
qa_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.1,
    top_p=0.9,
    max_tokens=512,
)
print("gpt-4o-mini initialized for reasoning, QA, and self-critique.")

# Vision Model: same gpt-4o-mini instance handles image inputs natively.
# Slightly higher max_tokens for richer captions.
vision_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.2,
    max_tokens=512,
)
print("gpt-4o-mini initialized for multi-modal image captioning.")

# Embedding Model: text-embedding-3-small
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)
print("text-embedding-3-small initialized for retrieval.")

print("\nAll OpenAI models ready.")

This configuration will:

Initialize the **gpt-4o-mini** model to function as both the primary generator and the self-critic by producing the reflection tokens (ISREL, ISSUP, and ISUSE). For the self-critique loop, the parameters are optimized for factual, deterministic, and stable answers.

Reuse **gpt-4o-mini** as the vision captioner. OpenAI's GPT-4o family natively understands images alongside text, so a single API removes the need to host a separate vision model locally.

Initialize the **text-embedding-3-small** model. This model generates the textual embeddings essential for efficient semantic search and retrieval in the FAISS vector store.

### Step 7. Load PDF documents from local disk

This step loads the source PDFs from your local filesystem into memory so they can be parsed in the next step. Place the PDF files you want to index inside a `data/` folder next to this notebook. This tutorial uses two example documents: the ICH E6(R3) Guideline and the EFPIA 2024 Annual Inspection Survey, but you can swap in any PDFs you'd like to query.

In [ ]:
# PDF Loading from Local Disk

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# List the PDFs you want to index. Adjust filenames as needed.
pdf_filenames = [
    "ICH_E6(R3)_Guideline.pdf",
    "inspection_survey.pdf",
]

pdf_files = {}
missing = []
for name in pdf_filenames:
    path = DATA_DIR / name
    if not path.exists():
        missing.append(str(path))
        continue
    with open(path, "rb") as f:
        pdf_files[name] = f.read()
    size_mb = len(pdf_files[name]) / (1024 * 1024)
    print(f"Loaded {name} ({size_mb:.2f} MB)")

if missing:
    raise FileNotFoundError(
        "The following PDFs were not found. Place them in "
        f"the '{DATA_DIR}/' directory and re-run this cell:\n  - "
        + "\n  - ".join(missing)
    )

print(f"\nAll {len(pdf_files)} PDFs loaded successfully.")

### Step 8. Multi-modal PDF parsing and captioning

This step is crucial for transforming our raw PDF documents into a multi-modal, searchable knowledge base for the self-RAG agent. 

In [ ]:
# Multi-Modal PDF Parsing and Captioning (OpenAI vision)
import os
import io
import pickle
import base64
from typing import List
from PIL import Image
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
import fitz  # PyMuPDF


def caption_image_with_openai(image: Image.Image) -> str:
    """Send a PIL image to gpt-4o-mini and return a descriptive caption."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    data_url = f"data:image/png;base64,{b64}"

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "Describe this image, chart, or diagram in detail. "
                    "Summarize its key findings or data points."
                ),
            },
            {"type": "image_url", "image_url": {"url": data_url}},
        ]
    )
    response = vision_llm.invoke([message])
    return response.content.strip() if isinstance(response.content, str) else str(response.content).strip()


def extract_and_caption_pdf(filename: str, pdf_content: bytes) -> List[Document]:
    """Extract text and images from in-memory PDF content, caption images, and return LangChain Documents."""
    print(f"\nProcessing {filename}...", flush=True)

    doc = fitz.open(stream=pdf_content, filetype="pdf")
    all_content = []

    # 1. Extract Text Chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    for i, page in enumerate(doc):
        text = page.get_text()
        chunks = text_splitter.split_text(text)
        for j, chunk in enumerate(chunks):
            doc_metadata = {"source": filename, "page": i + 1, "chunk_id": f"P{i+1}-T{j}"}
            all_content.append(Document(page_content=chunk, metadata=doc_metadata))

    # 2. Extract and Caption Images (only for image-heavy PDFs)
    if 'inspection_survey' in filename.lower():
        print(f"  -> {filename} identified as image-containing. Beginning image extraction...", flush=True)

        for i, page in enumerate(doc):
            image_list = page.get_images(full=True)
            for j, img_info in enumerate(image_list):
                try:
                    xref = img_info[0]
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]

                    image = Image.open(io.BytesIO(image_bytes))
                    if image.mode != 'RGB':
                        image = image.convert('RGB')

                    # Resize huge images to control payload size sent to the API
                    MAX_DIM = 1024
                    if max(image.size) > MAX_DIM:
                        image.thumbnail((MAX_DIM, MAX_DIM), Image.Resampling.LANCZOS)

                    print(f"    -> Captioning image {j+1} on page {i+1}...", flush=True)
                    caption = caption_image_with_openai(image)

                    caption_doc = f"IMAGE CAPTION (Source: {filename}, Page {i+1}, Image {j+1}): {caption}"
                    img_metadata = {
                        "source": filename,
                        "page": i + 1,
                        "chunk_id": f"P{i+1}-I{j}",
                        "type": "image_caption",
                    }
                    all_content.append(Document(page_content=caption_doc, metadata=img_metadata))

                except Exception as e:
                    print(f"    Error processing image on page {i+1}, image {j+1}: {e}", flush=True)
                    continue

    return all_content


# Execution of the Multi-modal Parsing (with caching to avoid repeated OpenAI calls)
CACHE_FILE = 'multimodal_documents_cache.pkl'
all_documents = []

if os.path.exists(CACHE_FILE):
    print(f"\nCache file found: {CACHE_FILE}. Loading documents from cache...", flush=True)
    try:
        with open(CACHE_FILE, 'rb') as f:
            all_documents = pickle.load(f)
        print("Documents successfully loaded from cache. Skipping multi-modal parsing.", flush=True)
    except Exception as e:
        print(f"Error loading cache file: {e}. Attempting to run full parsing.", flush=True)
        os.remove(CACHE_FILE)

if not all_documents:
    print(f"\nRunning Multi-Modal PDF Parsing and Captioning...", flush=True)
    for filename, content in pdf_files.items():
        all_documents.extend(extract_and_caption_pdf(filename, content))

    print(f"\nFinished parsing. Total documents created: {len(all_documents)}", flush=True)
    try:
        with open(CACHE_FILE, 'wb') as f:
            pickle.dump(all_documents, f)
        print(f"Successfully saved all {len(all_documents)} documents to {CACHE_FILE}.", flush=True)
    except Exception as e:
        print(f"WARNING: Could not save cache file {CACHE_FILE}: {e}", flush=True)

print(f"\nTotal documents (text chunks + image captions) available: {len(all_documents)}", flush=True)

This parsing will:

•	Define the function and use `fitz` to accurately pull both text and embedded image bytes from structured documents, a task simple text readers often fail at.

•	Base64-encode each extracted image and send it as a multi-modal message to OpenAI's `gpt-4o-mini`, which returns a textual caption. By converting images into descriptive text captions, we make visual information searchable via the standard text embedding model. This mechanism ensures the agent is not "blind" to non-textual context, thus improving the completeness of the knowledge base.

•	Implement caching logic to store the results, preventing the time-consuming and API-cost-intensive multi-modal captioning process from having to be repeated. Storing the processed knowledge base speeds up development and repeated execution.

•	Ensure the final knowledge base gives the self-reflective agent full context that includes both textual and visual data. This is the main objective of the entire process, giving the later self-reflective retrieval the foundation it needs to be precise and well-founded.

### Step 9. Indexing and retriever setup

This step completes the preparation of the multi-modal knowledge base by indexing all processed document chunks into an efficient, searchable vector store, which forms the basis for the agent's initial retrieval capability.

In [ ]:
# Indexing and Retriever Setup
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("\nStarting Vector Store Creation", flush=True)

try:
    # Create the FAISS Vector Store
    vectorstore = FAISS.from_documents(
        documents=all_documents,
        embedding=embeddings_model,
    )
    print(f"Vector Store created successfully with {len(all_documents)} documents.", flush=True)

    # We set 'k=5' to retrieve the top 5 most similar documents for any given query.
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print("Retriever configured (k=5). Ready for RAG.", flush=True)

except Exception as e:
    print(f"Vector Store creation failed: {e}", flush=True)

This configuration plays a key role in preparing the retrieval layer for the self-RAG workflow:

•	It builds a high efficiency vector store using FAISS, which is well known for its speed and scalability when handling dense vector indexes. This ensures that similarity searches run quickly which is critical for maintaining a responsive RAG pipeline.

•	It transforms the multi modal knowledge base into vector representations, allowing the retriever to match user queries by meaning rather than relying on exact keyword overlap.

•	It fine tunes context delivery by typically retrieving the top five most relevant documents (k=5), balancing precision and relevance within the model’s context window.

•	It establishes a single, consistent knowledge source that the self RAG agent can depend on for factual grounding that is an essential element of any trustworthy retrieval augmented system.

### Step 10. LangGraph state and core self-RAG logic

This step sets up the main sections of the self-RAG workflow. The agent state tracks the entire process. The LangGraph node functions manage the flexible, self-correcting logic.

In [ ]:
# LangGraph state and core self-RAG logic

from typing import TypedDict, List
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END
import re
# Assumed objects: qa_llm (ChatOpenAI), retriever, calculate_score (defined below)


def llm_text(prompt: str) -> str:
    """Invoke the ChatOpenAI model and return its plain-text content."""
    response = qa_llm.invoke(prompt)
    content = getattr(response, "content", response)
    return content if isinstance(content, str) else str(content)


# Define the Agent State (Schema)
class AgentState(TypedDict):
    """Represents the state of the Self-RAG agent."""
    query: str
    retrieved_docs: List[Document]
    generation_history: List[str]
    critique_score: float
    segment_count: int
    finish_generation: bool


# LangGraph Node Functions (SELF-RAG Logic)
MAX_SEGMENTS = 10
SCORE_THRESHOLD = 2.5  # used in evaluate_critique


def calculate_score(isrel_val: str, issup_val: str, isuse_val: int) -> float:
    """Calculate the combined weighted score for a segment (Soft Constraint)."""
    W_ISSUP = 3.0
    W_ISREL = 1.5
    W_ISUSE = 0.5

    score_rel = 1.0 if isrel_val == "Relevant" else 0.0

    if "Fully Supported" in issup_val:
        score_sup = 1.0
    elif "Partially" in issup_val:
        score_sup = 0.5
    else:
        score_sup = 0.0

    score_use = (isuse_val - 1) / 4.0

    total_score = (W_ISREL * score_rel) + (W_ISSUP * score_sup) + (W_ISUSE * score_use)
    return total_score


def initial_decision(state: AgentState) -> AgentState:
    """Initial decision on whether to retrieve based on the query type."""
    query = state["query"]

    prompt = f"""
    You are an expert self-reflecting LLM. Your task is to determine if external knowledge is required to answer the following query accurately.
    - If knowledge is required, output the token: <|Retrieve=Yes|>
    - If the query is open-ended or based on common knowledge, output the token: <|Retrieve=No|>

    Query: \"{query}\"

    Decision Token:
    """

    response = llm_text(prompt)

    if "<|Retrieve=Yes|>" in response:
        print("Decision: Retrieval required.", flush=True)
        return {"query": query, "retrieved_docs": [], "critique_score": 0.0, "segment_count": 0, "finish_generation": False}
    else:
        print("Decision: No retrieval required for initial generation.", flush=True)
        return {"query": query, "retrieved_docs": [Document(page_content="No documents retrieved.")], "critique_score": 0.0, "segment_count": 0, "finish_generation": False}


def retrieve_docs(state: AgentState) -> AgentState:
    """Retrieve documents based on the current query or the last generated segment."""
    query = state["query"]

    if state.get("generation_history"):
        search_query = state["generation_history"][-1]
    else:
        search_query = query

    print(f"Retrieving documents for: '{search_query[:50]}...'", flush=True)
    docs = retriever.invoke(search_query)

    return {"retrieved_docs": docs}


def generate_segment(state: AgentState) -> AgentState:
    """Generate the next answer segment and self-reflect using critique tokens."""
    query = state["query"]
    history = state.get("generation_history", [])

    docs_context = "\n---\n".join(
        [f"Source ({d.metadata.get('chunk_id')}): {d.page_content}" for d in state["retrieved_docs"]]
    )
    history_context = "\n".join(history)

    prompt = f"""
    You are a SELF-RAG agent powered by an OpenAI GPT-4o model. Your goal is to generate one accurate, concise segment of an answer.

    INSTRUCTION: Generate a comprehensive, multi-segment answer to the user's query.
    1. CONTEXT: Use the provided document segments (which include text and image captions) to answer the question accurately.
    2. SEGMENTATION: Only use the <|END|> token when the answer is fully comprehensive and detailed, and you have no more relevant information to add.
    3. REFLECTION: After generating the segment, immediately append these key-value reflection tokens:
        - ISREL: <|ISREL=Relevant|> or <|ISREL=Irrelevant|>
        - ISSUP: <|ISSUP=Fully Supported|> or <|ISSUP=Partially Supported|> or <|ISSUP=No Support|>
        - ISUSE: <|ISUSE=N|> (where N is the overall quality/utility score from 1 to 5, 5 is best).

    CURRENT QUERY: \"{query}\"

    HISTORY SO FAR: \"{history_context}\"

    RETRIEVED CONTEXT (Multi-Modal: text chunks and image captions):
    {docs_context}

    ---

    Generate the NEXT SEGMENT and REFLECTION TOKENS. End the entire generation with <|END|> if the answer is complete.
    """

    print(f"Generating Segment {state['segment_count'] + 1}...", flush=True)
    full_response = llm_text(prompt)

    CRITIQUE_TOKENS = ["<|ISREL=", "<|ISSUP=", "<|ISUSE=", "|>"]

    isrel = re.search(r"<\|ISREL=(.+?)\|>", full_response)
    issup = re.search(r"<\|ISSUP=(.+?)\|>", full_response)
    isuse = re.search(r"<\|ISUSE=(\d+)\|>", full_response)

    isrel_val = isrel.group(1).strip() if isrel else "Irrelevant"
    issup_val = issup.group(1).strip() if issup else "No Support"
    isuse_val = int(isuse.group(1).strip()) if isuse and isuse.group(1).isdigit() else 1

    segment = full_response
    for token in CRITIQUE_TOKENS + ["<|Retrieve=Yes|>", "<|Retrieve=No|>", "<|END|>"]:
        segment = segment.replace(token, "").strip()

    new_history = history + [segment]

    print(f"  -> ISREL: {isrel_val}, ISSUP: {issup_val}, ISUSE: {isuse_val}", flush=True)

    return {
        "generation_history": new_history,
        "segment_count": state["segment_count"] + 1,
        "finish_generation": "<|END|>" in full_response,
        "critique_score": calculate_score(isrel_val, issup_val, isuse_val),
        "retrieved_docs": state["retrieved_docs"],
    }


def evaluate_critique(state: AgentState) -> str:
    """Conditional edge function to determine the next step based on critique score."""
    score = state["critique_score"]
    segment_count = state["segment_count"]
    is_finished = state["finish_generation"]

    if is_finished or segment_count >= MAX_SEGMENTS:
        return "end"

    if score < SCORE_THRESHOLD:
        print(f"Critique: Low score ({score:.2f}) observed. FORCING RE-RETRIEVAL for next segment.", flush=True)
        return "retrieve"

    print(f"Critique: High score ({score:.2f}) observed. Continuing generation.", flush=True)
    return "continue"


def finalize_answer(state: AgentState) -> AgentState:
    """Compile the final answer."""
    final_answer = "\n".join(state["generation_history"])
    print("\n--- FINAL ANSWER ---", flush=True)
    print(final_answer, flush=True)
    return state


# Build and Compile the LangGraph Workflow
print("\nBuilding and Compiling LangGraph Workflow", flush=True)

workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("initial_decision", initial_decision)
workflow.add_node("retrieve_docs", retrieve_docs)
workflow.add_node("generate_segment", generate_segment)
workflow.add_node("finalize_answer", finalize_answer)

# Define Edges (Flow Control)
workflow.set_entry_point("initial_decision")

workflow.add_conditional_edges(
    "initial_decision",
    lambda state: "retrieve" if not state["retrieved_docs"] else "generate",
    {
        "retrieve": "retrieve_docs",
        "generate": "generate_segment",
    },
)

workflow.add_edge("retrieve_docs", "generate_segment")

workflow.add_conditional_edges(
    "generate_segment",
    evaluate_critique,
    {
        "retrieve": "retrieve_docs",
        "continue": "generate_segment",
        "end": "finalize_answer",
    },
)

workflow.add_edge("finalize_answer", END)

app = workflow.compile()
print("LangGraph workflow compiled successfully (object named 'app').", flush=True)

This code serves several purposes:

•	The agent keeps a core memory that stores its evolving response, the evidence it has retrieved, and internal feedback. This memory helps the agent's logic to dynamically improve its reasoning by storing context across various steps.

•	 The agent first determines whether adequate factual grounding is present before producing any segments. To ensure that the generated response is accurate and pertinent, the agent intelligently seeks for stronger, more supportive information if the existing context is deemed incomplete.

•	Alongside each generated segment, the model issues internal reflection tokens that immediately quantify the output's relevance, factual support, and overall quality. These critical signals are then combined into a single critique score, giving the agent an objective, measurable way to judge its own performance.

•	Determined by the critique score, the agent then decides whether to rework, expand upon, or finalize its answer. This iterative process makes the system inherently resilient, forcing it to improve incorrect generations and maintain factual precision over multiple reasoning rounds.

### Step 11. LangGraph state and core self-RAG logic

The entire self-RAG workflow begins with this last step.

In [ ]:
# Execute the LangGraph Workflow

# 1. Define the Query
# This query is designed to require information from both documents.
user_query = "What is the primary purpose of the ICH E6(R3) Guideline and what are the key findings from the EFPIA 2024 inspection survey regarding remote inspections?"

# 2. Define the Initial Input State
# The generation_history must start empty.
inputs = {
    "query": user_query, 
    "generation_history": []
}

print(f"\n--- STARTING LANGGRAPH EXECUTION ---", flush=True)
print(f"Query: {user_query}\n", flush=True)

# 3. Stream the Execution
# This loop runs the graph and prints the state update after each node completes.
for step in app.stream(inputs):
    # Print the name of the node that just executed and its resulting state
    print(step, flush=True)
    print("\n--- NODE TRANSITION ---", flush=True) 

print(f"--- LANGGRAPH EXECUTION COMPLETE ---", flush=True)

In [ ]:
# Final Answer Extraction and Review


final_state = None
# The query and inputs are reused from Step 10
user_query = "What is the primary purpose of the ICH E6(R3) Guideline and what are the key findings from the EFPIA 2024 inspection survey regarding remote inspections?"
inputs = {"query": user_query, "generation_history": []}


print("\n--- RE-RUNNING EXECUTION FOR FINAL EXTRACTION ---", flush=True)

for step in app.stream(inputs):
    for key, value in step.items():
        # The key tells us which node just ran (e.g., 'finalize_answer')
        # The value is the state output of that node
        if key == "finalize_answer":
            final_state = value 
        elif key == END:
            # If the END node is hit, the graph is finished
            final_state = value 

# 2. Extract and Format the Final Answer
if final_state and "generation_history" in final_state:
    # Join all generated segments into one cohesive answer
    final_answer = "\n".join(final_state["generation_history"]).strip()

    print("\n==============================================", flush=True)
    print(" RAG PIPELINE COMPLETE", flush=True)
    print("==============================================", flush=True)
    print(f"USER QUERY:\n{user_query}\n", flush=True)
    print(f"FINAL GENERATED ANSWER ({final_state['segment_count']} segments):", flush=True)
    print("----------------------------------------------", flush=True)
    print(final_answer, flush=True)
    print("----------------------------------------------", flush=True)
else:
    print("\n EXECUTION FAILED or final state was not captured.", flush=True)
    print(f"Last recorded state: {final_state}", flush=True)

Once the agent either hits the maximum number of segments or completes its multi-segment answer, it produces the final output to the user question. The `.stream()` method is then used to run the compiled graph, represented by the app object.

The initial state, which contains the detailed `user_query`, is passed in through the inputs dictionary.

As the graph streams, each loop processes one node at a time based on the system’s internal logic. Every node’s output is printed as it runs, letting us watch the agent refine its reasoning in real time and build its multi-part response ultimately ending with a well-supported final answer. The final step reruns the full self-RAG workflow to create a refined answer. It executes the LangGraph and watches the streaming state updates until the `finalize_answer` or `END` node shows up. It pulls the generated segments and joins them into a grounded final answer whenever the final state is reached.


In [ ]:

# 1. Define the query and inputs for the graph execution
user_query = "What does the ICH ER6 guideline say about Quality Assurance and Quality Control?"
inputs = {"query": user_query, "generation_history": []}

# 2. Rerun stream and capture the final state
print("--- Rerunning stream to answer the new query ---", flush=True)

final_state = None
# This loop runs the graph and prints the state update after each node completes.
for step in app.stream(inputs):
    print(step, flush=True)
    print("\n--- NODE TRANSITION ---\n", flush=True)
    # Capture the last yielded state (which contains the compiled final history)
    for key, value in step.items():
        if key != END:
            final_state = value

# 3. Extract and Format the Final Answer
if final_state and "generation_history" in final_state:
    # Join all generated segments (which should now be clean)
    final_answer_text = "\n".join(final_state["generation_history"]).strip()
    
    # Run a final cleanup pass
    final_answer_text = re.sub(r'\s*(Relevant|Irrelevant)\s*(Fully Supported|Partially Supported|No Support)\s*\d', '', final_answer_text).strip()
    final_answer_text = final_answer_text.replace("<|END", "").strip()

    # 4. Present the Results
    print("\n\n#####################################################", flush=True)
    print("            FINAL SELF-RAG ANSWER                 ", flush=True)
    print("#####################################################\n", flush=True)

    print("--- ANSWER ---", flush=True)
    print(final_answer_text, flush=True)
    print("\n#####################################################", flush=True)

else:
    print("\n EXECUTION FAILED or final state was not captured.", flush=True)


In [ ]:
# USER QUERY: According to EFPIA 2024 data on multiple inspections at manufacturing sites, which countries recorded the highest inspection counts per site, and what does this reveal about their regulatory significance?
# 1. Define the query and inputs for the graph execution
user_query = "According to EFPIA 2024 data on multiple inspections at manufacturing sites, which countries recorded the highest inspection counts per site, and what does this reveal about their regulatory significance?"
inputs = {"query": user_query, "generation_history": []}

# 2. Rerun stream and capture the final state
print("--- Rerunning stream to answer the new combined query ---", flush=True)

final_state = None
# This loop runs the graph and prints the state update after each node completes.
for step in app.stream(inputs):
    print(step, flush=True)
    print("\n--- NODE TRANSITION ---\n", flush=True)
    # Capture the last yielded state (which contains the compiled final history)
    for key, value in step.items():
        if key != END:
            final_state = value

# 3. Extract and Format the Final Answer
if final_state and "generation_history" in final_state:
    # Join all generated segments (which should now be clean)
    final_answer_text = "\n".join(final_state["generation_history"]).strip()
    
    # Run a final cleanup pass
    final_answer_text = re.sub(r'\s*(Relevant|Irrelevant)\s*(Fully Supported|Partially Supported|No Support)\s*\d', '', final_answer_text).strip()
    final_answer_text = final_answer_text.replace("<|END", "").strip()

    # 4. Present the Results
    print("\n\n#####################################################", flush=True)
    print("            FINAL SELF-RAG ANSWER                  ", flush=True)
    print("#####################################################\n", flush=True)

    print("--- ANSWER ---", flush=True)
    print(final_answer_text, flush=True)
    print("\n#####################################################", flush=True)

else:
    print("\n EXECUTION FAILED or final state was not captured.", flush=True)

The self-reflective retrieval augmented generation setup in this tutorial offers major advantages over standard RAG, mainly in terms of reliability and smart efficiency. Its biggest strength is improved factual accuracy and traceability, made possible by the OpenAI `gpt-4o-mini` model running its own self-critiques with reflection tokens. These critiques produce a score that guides the workflow, allowing adaptive retrieval; the model only pulls new context when a segment isn't well supported. This approach also makes it easier to work with complex, multi-modal documents, since image captions produced by the same `gpt-4o-mini` vision-capable model can be added to the vector store. The result is a more trustworthy, flexible query agent that checks and segments its answers against the knowledge base before giving the final result.